# 08 — Mammo-FM Downstream

One directly runnable notebook represents one architecture. Change only CONDITION and SEED to cover its 12 protocol jobs.

## 1. Experiment configuration

In [ ]:
import os
from pathlib import Path
import sys

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'configs').is_dir())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'notebooks/utility'))

from notebooks.utility.downstream_experiment import (
    build_error_case_table,
    configure_environment,
    construct_dataset,
    experiment_configuration,
    load_adapter,
    load_existing_outputs,
    load_prediction_rows,
    plot_calibration,
    plot_source_accounting,
    plot_training_history,
    plot_validation_curves,
    run_validation,
    train,
)

ARCHITECTURE = 'mammofm'
CONDITION = 'real_only'
SEED = 17
GPU_SELECTOR = os.environ.get('MAMMODIFFUSION_GPU')

if not GPU_SELECTOR:
    raise RuntimeError('Set MAMMODIFFUSION_GPU to a physical GPU index or a GPU UUID.')

os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
MAMMOFM_LOCAL_CHECKPOINT_PATH = os.environ.get('MAMMOFM_LOCAL_CHECKPOINT_PATH')

configuration = experiment_configuration(
    ROOT, ARCHITECTURE, CONDITION, SEED, gpu=GPU_SELECTOR
)
configuration['root'] = str(ROOT)
configuration


## 2. Environment and GPU

In [ ]:
environment = configure_environment(configuration)
configuration['gpu_uuid'] = environment['resolved_uuid']
configuration['gpu_name'] = environment['observed_name']
configuration['gpu_physical_index'] = environment['resolved_physical_index']
environment

## 3. Dataset construction

In [ ]:
dataset = construct_dataset(ROOT, configuration)
len(dataset['train_rows']), len(dataset['validation_rows'])

## 4. Dataset audit

In [ ]:
dataset['audit']

## 5. Training

In [ ]:
training_result = train(ROOT, configuration, dataset)

## 6. Training curves

In [ ]:
existing_outputs = load_existing_outputs(ROOT, configuration)
training_history_figure = (
    plot_training_history(existing_outputs['history'])
    if existing_outputs['history']
    else None
)
source_accounting_figure = (
    plot_source_accounting(existing_outputs['source_accounting'])
    if existing_outputs['source_accounting']
    else None
)
{
    'training_curves': training_history_figure,
    'source_accounting': existing_outputs['source_accounting'],
    'source_accounting_figure': source_accounting_figure,
}


## 7. Best checkpoint

In [ ]:
existing_outputs = load_existing_outputs(ROOT, configuration)
checkpoint = existing_outputs["checkpoint"]

if checkpoint is None:
    raise RuntimeError("No trained checkpoint is available.")

checkpoint


## 8. Validation inference

In [ ]:
existing_outputs = load_existing_outputs(ROOT, configuration)
checkpoint = existing_outputs["checkpoint"]

if checkpoint is None:
    raise RuntimeError("No trained checkpoint is available.")

validation_result = run_validation(
    ROOT,
    configuration,
    dataset,
    checkpoint,
)
validation_rows = validation_result['rows']
validation_metrics = validation_result['metrics']
validation_result


## 9. Validation metrics

In [ ]:
validation_curve_figure = plot_validation_curves(
    validation_rows, validation_metrics['threshold']
)
validation_metrics


## 10. Calibration

In [ ]:
calibration_figure = plot_calibration(validation_rows)
calibration_figure

## 11. Error analysis

In [ ]:
error_cases = build_error_case_table(validation_rows, validation_metrics['threshold'])
error_tables = ({'false positives': error_cases[error_cases.error_type == 'false_positive'],
                 'false negatives': error_cases[error_cases.error_type == 'false_negative'],
                 'highest-confidence correct predictions': error_cases[error_cases.error_type == 'correct']}
)
from notebooks.utility.classifier_interpretability import largest_ft_fs_disagreements
ft_rows = load_prediction_rows(ROOT / 'results/publication_v2/downstream/mammofm/real_plus_best_finetuned_positive' / f'seed_{SEED}/validation_predictions.csv')
fs_rows = load_prediction_rows(ROOT / 'results/publication_v2/downstream/mammofm/real_plus_best_fromscratch_positive' / f'seed_{SEED}/validation_predictions.csv')
largest_disagreements = largest_ft_fs_disagreements(ft_rows, fs_rows) if ft_rows and fs_rows else None
error_tables['largest FT-vs-FS disagreements'] = largest_disagreements
error_tables

## 12. Interpretability

In [ ]:
from notebooks.utility.classifier_interpretability import (
    mammofm_attribution,
    preregistered_cases,
)
import numpy as np

interpretability_cases = preregistered_cases(
    validation_rows, validation_metrics['threshold']
)
if not interpretability_cases:
    raise RuntimeError('Deterministic validation cases are required for interpretability.')

adapter = load_adapter(configuration)
model = adapter.load_checkpoint(checkpoint)
case_loader = adapter.build_validation_dataloader(interpretability_cases, seed=SEED)
attribution_root = Path(configuration['results_dir']) / 'interpretability'
attribution_root.mkdir(parents=True, exist_ok=True)
case_index = 0
for images, _labels in case_loader:
    for image in images:
        category = interpretability_cases[case_index]['category']
        heatmap = mammofm_attribution(model, image.unsqueeze(0))
        np.save(attribution_root / f'{category}_{case_index}.npy', heatmap)
        case_index += 1

{'method': 'EfficientNet spatial-feature attribution', 'cases': interpretability_cases}
